# Week 7 (starter): The Adaptation Decision and a Production Dataset

No GPU, no fine-tuning. The dataset and the dynamic few-shot selection run locally; the evaluation calls Gemini (guarded: without a key it reports API-BLOCKED, the local parts still run). Cells marked **TODO (you)** are yours. Dependencies: `tiktoken`, `sentence-transformers`. For the eval: `pip install openai`.

In [1]:
import os, re, json, random, hashlib, statistics, pathlib, time
from collections import Counter, defaultdict
from IPython.display import display, Markdown
random.seed(7)
ROOT = pathlib.Path(".")
DATA, RESULTS = ROOT / "data", ROOT / "results"
DATA.mkdir(exist_ok=True); RESULTS.mkdir(exist_ok=True)
# Customer-support ticket classification
TEMPLATES = {
    "billing_issue": [
        "I was charged {amount} twice for {product}",
        "My {period} invoice has an incorrect {field}",
        "Why is there a {amount} charge on my account?",
        "I was billed {amount} after cancelling {product}",
        "Please correct the {field} on my {period} statement",
        "My payment for {product} was {problem}",
        "I need an explanation of the {amount} fee",
        "The invoice for {product} contains an unexpected charge",
        "My account balance is wrong because of {field}",
        "Please review my recent payment of {amount}",
        "I don't recognize the {amount} charge for {product}",
        "The total due on my {period} bill is incorrect"
    ],
    "technical_problem": [
        "The {product} app {problem} when I {action}",
        "I cannot {action} in the {product} app",
        "The {feature} feature is {problem}",
        "I receive error {error} when trying to {action}",
        "After the update, {product} {problem}",
        "My {device} cannot run {product} correctly",
        "The {feature} page keeps {problem}",
        "A technical error prevents me from {action}",
        "Error {error} appears whenever I open {feature}",
        "The {product} application fails during {action}",
        "My {device} freezes when I use {feature}",
        "Please troubleshoot the {problem} in {product}"
    ],
    "account_access": [
        "I cannot sign in using {method} on my {device}",
        "My password reset link {problem}",
        "The login code sent by {channel} {problem}",
        "My account was locked after {attempts} failed attempts",
        "I need to recover my {account} account using {method}",
        "Two-factor authentication is failing on my {device}",
        "I cannot verify my identity through {channel}",
        "My login keeps failing with {method}",
        "I lost access to my registered {channel}",
        "Please unlock my account after {attempts} failed logins",
        "I cannot update my {field} because I cannot sign in",
        "Account recovery through {channel} is not working"
    ],
    "refund_request": [
        "Please refund my {product} purchase made {period}",
        "I returned {product} and want my payment back",
        "My refund for {product} has been pending for {days} days",
        "I was promised a refund for order {order}",
        "How can I request a refund for {product}?",
        "I want my money back because {reason}",
        "Please reverse the payment of {amount} for {product}",
        "My refund has not arrived after {days} days",
        "I need a refund for order {order} because {reason}",
        "Can you check my {product} refund status?",
        "I was charged {amount} and want it refunded",
        "Please refund the returned {product}"
    ],
    "shipping_delivery": [
        "Where is order {order}? It was expected {period}",
        "My {product} delivery is {problem}",
        "Tracking for package {order} has not updated for {days} days",
        "My delivery arrived damaged and contained {product}",
        "Order {order} says delivered but I cannot find it",
        "Please change the shipping address for order {order}",
        "My shipment is delayed because {reason}",
        "I received the wrong {product} in my package",
        "What is the estimated arrival date for order {order}?",
        "My package has been {problem} since {period}",
        "The delivery status for order {order} is incorrect",
        "Help me track my shipment delayed for {days} days"
    ],
    "cancellation_request": [
        "Please cancel my {product} subscription effective {period}",
        "I want to stop automatic renewal of {product}",
        "Please terminate my {product} plan because {reason}",
        "How do I cancel my {period} membership?",
        "I no longer want {product}; end my subscription",
        "Cancel the upcoming renewal scheduled for {period}",
        "Please close my {account} account and cancel the plan",
        "I want to discontinue {product} before the next bill",
        "Confirm my {product} subscription will not renew {period}",
        "Please stop recurring charges for {product}",
        "I need to end my {period} service because {reason}",
        "Cancel my membership and confirm the end date"
    ]
}

SLOTS = {
    "amount": ["$12", "$25", "$49", "$80", "$120", "$15.99", "$39.99", "$7.50", "$65", "$200", "$18", "$95"],
    "product": ["premium plan", "mobile app", "annual subscription", "headphones", "software license", "online course", "wireless speaker", "monthly plan", "fitness tracker", "cloud storage", "smartwatch", "printer", "tablet", "business account"],
    "field": ["tax", "discount", "total", "billing date", "subscription fee", "payment amount", "account balance", "service charge", "invoice number"],
    "problem": ["fails", "freezes", "shows an error", "keeps timing out", "is unavailable", "is incorrect", "was rejected", "never loads", "keeps restarting", "is stuck", "does not respond"],
    "period": ["yesterday", "last week", "last month", "Monday", "the weekend", "my last billing cycle", "three days ago", "the last renewal", "this morning", "two weeks ago"],
    "action": ["save settings", "download a report", "submit a form", "sync files", "complete checkout", "upload documents", "open the dashboard", "update my profile"],
    "feature": ["search", "notifications", "dashboard", "file upload", "checkout", "profile editor", "report export", "account settings"],
    "error": ["104", "403", "500", "E21", "A17", "502", "E08", "C19", "301", "504"],
    "device": ["laptop", "phone", "tablet", "desktop", "work computer", "Android device", "iPhone", "Windows PC"],
    "method": ["my saved password", "email login", "single sign-on", "my username", "the mobile app", "my security key"],
    "channel": ["email", "phone number", "authenticator app", "backup address", "SMS", "recovery email"],
    "attempts": ["3", "5", "7", "10", "12", "15", "20", "25"],
    "account": ["customer", "business", "personal", "work", "main", "company", "online"],
    "days": ["2", "3", "5", "7", "10", "14", "21", "28", "30"],
    "order": ["10482", "20751", "31864", "42590", "58321", "67104", "74219", "85630", "96315", "51428", "62387", "73519"],
    "reason": ["the item arrived damaged", "the service did not work", "I ordered the wrong item", "the delivery was late", "I no longer need it", "the product was not as described", "the feature is missing", "the item arrived too late"]
}

def fill_template(template):
    return re.sub(r"\{(\w+)\}", lambda m: random.choice(SLOTS[m.group(1)]), template)

raw = [
    {"text": fill_template(random.choice(templates)), "label": label}
    for label, templates in TEMPLATES.items()
    for _ in range(2000)
]
raw += [{"text": "", "label": "billing_issue"}] * 5
raw += [{"text": "hello", "label": "unknown"}] * 4
VALID = set(TEMPLATES)
valid = [r for r in raw if isinstance(r.get("text"), str) and r["text"].strip() and r.get("label") in VALID]
seen, clean = set(), []
for r in valid:
    key = (re.sub(r"\s+", " ", r["text"].strip().casefold()), r["label"])
    if key not in seen:
        seen.add(key); clean.append({"text": r["text"].strip(), "label": r["label"]})
train, eval_ = [], []
for label in sorted(VALID):
    rows = [r for r in clean if r["label"] == label]
    random.shuffle(rows)
    if len(rows) < 51:
        raise ValueError(f"Only {len(rows)} unique examples for {label}; add more template variation.")
    train.extend(rows[:200]); eval_.extend(rows[200:210])
random.shuffle(train); random.shuffle(eval_)
train_texts = {re.sub(r"\s+", " ", r["text"].strip().casefold()) for r in train}
eval_texts = {re.sub(r"\s+", " ", r["text"].strip().casefold()) for r in eval_}
assert len(train) >= 200 and len(eval_) >= 50
assert train_texts.isdisjoint(eval_texts)
assert all(set(r) == {"text", "label"} and r["text"].strip() and r["label"] in VALID for r in train + eval_)
def save_jsonl(path, rows):
    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
save_jsonl(DATA / "train.jsonl", train); save_jsonl(DATA / "eval.jsonl", eval_)
print(f"Raw: {len(raw)} | Invalid removed: {len(raw)-len(valid)} | Duplicates removed: {len(valid)-len(clean)}")
print(f"Training: {len(train)} | Evaluation: {len(eval_)}")
print("Training labels:", dict(Counter(r["label"] for r in train)))
print("Evaluation labels:", dict(Counter(r["label"] for r in eval_)))

Raw: 12009 | Invalid removed: 9 | Duplicates removed: 8798
Training: 1200 | Evaluation: 60
Training labels: {'cancellation_request': 200, 'refund_request': 200, 'account_access': 200, 'billing_issue': 200, 'technical_problem': 200, 'shipping_delivery': 200}
Evaluation labels: {'billing_issue': 10, 'refund_request': 10, 'shipping_delivery': 10, 'technical_problem': 10, 'account_access': 10, 'cancellation_request': 10}


In [2]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
def quality_stats(rows):
    lengths = [len(enc.encode(r["text"])) for r in rows]
    return {
        "count": len(rows),
        "label_balance": dict(Counter(r["label"] for r in rows)),
        "token_min": min(lengths), "token_median": statistics.median(lengths),
        "token_mean": round(statistics.mean(lengths), 2), "token_max": max(lengths),
        "token_p90": sorted(lengths)[int(0.9 * (len(lengths)-1))],
        "empty_texts": sum(not r["text"].strip() for r in rows),
        "invalid_labels": sum(r["label"] not in VALID for r in rows),
        "schema_consistent": all(set(r) == {"text", "label"} for r in rows)
    }
quality = {
    "source": "Synthetic customer-support messages generated from documented templates and slot values",
    "raw_count": len(raw), "invalid_removed": len(raw) - len(valid),
    "duplicates_removed": len(valid) - len(clean),
    "train": quality_stats(train), "evaluation": quality_stats(eval_),
    "split_overlap": len({r["text"].casefold() for r in train} & {r["text"].casefold() for r in eval_}),
    "minimums_met": len(train) >= 200 and len(eval_) >= 50
}
(DATA / "quality_report.json").write_text(json.dumps(quality, indent=2), encoding="utf-8")
print(json.dumps(quality, indent=2))

{
  "source": "Synthetic customer-support messages generated from documented templates and slot values",
  "raw_count": 12009,
  "invalid_removed": 9,
  "duplicates_removed": 8798,
  "train": {
    "count": 1200,
    "label_balance": {
      "cancellation_request": 200,
      "refund_request": 200,
      "account_access": 200,
      "billing_issue": 200,
      "technical_problem": 200,
      "shipping_delivery": 200
    },
    "token_min": 5,
    "token_median": 10.0,
    "token_mean": 9.81,
    "token_max": 16,
    "token_p90": 12,
    "empty_texts": 0,
    "invalid_labels": 0,
    "schema_consistent": true
  },
  "evaluation": {
    "count": 60,
    "label_balance": {
      "billing_issue": 10,
      "refund_request": 10,
      "shipping_delivery": 10,
      "technical_problem": 10,
      "account_access": 10,
      "cancellation_request": 10
    },
    "token_min": 6,
    "token_median": 10.0,
    "token_mean": 10.05,
    "token_max": 16,
    "token_p90": 13,
    "empty_texts": 0,
 

## Part 2: Adapt with dynamic few-shot (local selection)
For each eval input, retrieve the most similar training examples to place in the prompt. This is the no-GPU adaptation, and it reuses Week 5 retrieval.

In [3]:
os.environ["HF_HOME"] = str(ROOT / ".hf_cache")
from sentence_transformers import SentenceTransformer, util
import numpy as np
emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
train_emb = emb.encode([r["text"] for r in train], convert_to_tensor=True, normalize_embeddings=True)
def few_shot(query, k=4):
    q_emb = emb.encode([query], convert_to_tensor=True, normalize_embeddings=True)
    sims = util.cos_sim(q_emb, train_emb)[0].cpu().numpy()
    return [train[i] for i in np.argsort(sims)[::-1][:k]]
example = eval_[0]
print("Query:", example["text"], "| True label:", example["label"])
for s in few_shot(example["text"]):
    print(s["text"], "->", s["label"])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Query: I don't recognize the $65 charge for annual subscription | True label: billing_issue
I don't recognize the $80 charge for annual subscription -> billing_issue
I don't recognize the $95 charge for annual subscription -> billing_issue
I was charged $49 twice for annual subscription -> billing_issue
I was charged $39.99 twice for annual subscription -> billing_issue


#Part 3

In [4]:
# Gemini import
from openai import OpenAI
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [5]:
import os, json
from openai import OpenAI
from google.colab import userdata

MODEL = "gemini-3.5-flash-lite"
OUT = RESULTS / "evaluation_results.jsonl"
CACHE = RESULTS / "evaluation_cache.json"
api_key = userdata.get("GEMINI_API_KEY")
if not api_key: raise RuntimeError("API-BLOCKED: Set GEMINI_API_KEY in Colab Secrets.")

client = OpenAI(api_key=api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/", max_retries=0, timeout=120.0)

def gemini_chat(messages, model=MODEL):
    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

# One ticket per category = 6 tickets, 12 calls maximum
eval_subset = []
for label in sorted(VALID):
    eval_subset.extend([r for r in eval_ if r["label"] == label][:1])
if len(eval_subset) != 6: raise ValueError(f"Expected 6 tickets, got {len(eval_subset)}.")

cache = json.loads(CACHE.read_text(encoding="utf-8")) if CACHE.exists() else {}
valid_lower = {label.lower(): label for label in VALID}

def normalize_prediction(pred):
    if not isinstance(pred, str): return "invalid"
    return valid_lower.get(pred.strip().lower().strip("`'\"* ."), "invalid")

def save_cache():
    CACHE.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")

results = []
for i, row in enumerate(eval_subset):
    shots = few_shot(row["text"], k=4)
    record = {"id": i, "text": row["text"], "true_label": row["label"], "retrieved_examples": shots}

    for condition in ["zero_shot", "few_shot"]:
        key = json.dumps({"model": MODEL, "condition": condition, "text": row["text"], "shots": [{"text": s["text"], "label": s["label"]} for s in shots] if condition == "few_shot" else []}, sort_keys=True)

        if key in cache:
            pred = cache[key]
            print(f"Using cached {condition} result for ticket {i+1}")
        else:
            examples = "".join(f"Ticket: {s['text']}\nCategory: {s['label']}\n" for s in shots) if condition == "few_shot" else ""
            prompt = f"Classify this customer-support ticket into exactly one category: {', '.join(sorted(VALID))}. Return only the category label.\n{examples}Ticket: {row['text']}\nCategory:"
            print(f"Calling Gemini for ticket {i+1}: {condition}")
            pred = normalize_prediction(gemini_chat([{"role": "user", "content": prompt}]))
            cache[key] = pred
            save_cache()

        record[f"{condition}_prediction"] = pred
        record[f"{condition}_correct"] = pred == row["label"]

    results.append(record)

save_jsonl(OUT, results)
n = len(results)
zero_acc = sum(r["zero_shot_correct"] for r in results) / n
few_acc = sum(r["few_shot_correct"] for r in results) / n
print(f"Evaluated: {n}")
print(f"Zero-shot accuracy: {zero_acc:.1%}")
print(f"Dynamic few-shot accuracy: {few_acc:.1%}")
print(f"Change: {(few_acc-zero_acc)*100:+.1f} percentage points")
print(f"Results saved to {OUT}; cache saved to {CACHE}")

Calling Gemini for ticket 1: zero_shot
Calling Gemini for ticket 1: few_shot
Calling Gemini for ticket 2: zero_shot
Calling Gemini for ticket 2: few_shot
Calling Gemini for ticket 3: zero_shot
Calling Gemini for ticket 3: few_shot
Calling Gemini for ticket 4: zero_shot
Calling Gemini for ticket 4: few_shot
Calling Gemini for ticket 5: zero_shot
Calling Gemini for ticket 5: few_shot
Calling Gemini for ticket 6: zero_shot
Calling Gemini for ticket 6: few_shot
Evaluated: 6
Zero-shot accuracy: 100.0%
Dynamic few-shot accuracy: 100.0%
Change: +0.0 percentage points
Results saved to results/evaluation_results.jsonl; cache saved to results/evaluation_cache.json


#Part 4

In [6]:
if (RESULTS / "evaluation_results.jsonl").exists():
    results = [json.loads(line) for line in (RESULTS / "evaluation_results.jsonl").read_text().splitlines() if line.strip()]
    def accuracy(rows, field):
        return sum(bool(r[field]) for r in rows) / len(rows) if rows else None
    zero_acc = accuracy(results, "zero_shot_correct")
    few_acc = accuracy(results, "few_shot_correct")
    print(f"Evaluated: {len(results)}")
    print(f"Zero-shot accuracy: {zero_acc:.1%}")
    print(f"Dynamic few-shot accuracy: {few_acc:.1%}")
    print(f"Change: {(few_acc-zero_acc)*100:+.1f} percentage points")
    slices = defaultdict(list)
    for r in results:
        slices[f"Category: {r['true_label']}"].append(r)
        slices["Short ticket (<=8 tokens)" if len(enc.encode(r["text"])) <= 8 else "Long ticket (>8 tokens)"].append(r)
        shots = r["retrieved_examples"]
        majority = Counter(s["label"] for s in shots).most_common(1)[0][0] if shots else None
        slices["Retrieved-label majority matches truth" if majority == r["true_label"] else "Retrieved-label majority differs from truth"].append(r)
    slice_report = []
    for name, rows in slices.items():
        z, f = accuracy(rows, "zero_shot_correct"), accuracy(rows, "few_shot_correct")
        item = {"slice": name, "n": len(rows), "zero_shot_accuracy": round(z, 4), "few_shot_accuracy": round(f, 4), "change_pp": round((f-z)*100, 2)}
        slice_report.append(item)
        print(item)
    underperforming = [s for s in slice_report if s["few_shot_accuracy"] < s["zero_shot_accuracy"]]
    if underperforming:
        worst = min(underperforming, key=lambda s: s["change_pp"])
        print("\nUnderperforming slice:", worst)
        print("Inspect the following errors:")
        for r in results:
            shots = r["retrieved_examples"]
            majority = Counter(s["label"] for s in shots).most_common(1)[0][0] if shots else None
            is_slice = (
                worst["slice"] == f"Category: {r['true_label']}" or
                (worst["slice"] == "Short ticket (<=8 tokens)" and len(enc.encode(r["text"])) <= 8) or
                (worst["slice"] == "Long ticket (>8 tokens)" and len(enc.encode(r["text"])) > 8) or
                (worst["slice"] == "Retrieved-label majority matches truth" and majority == r["true_label"]) or
                (worst["slice"] == "Retrieved-label majority differs from truth" and majority != r["true_label"])
            )
            if is_slice and r["zero_shot_correct"] and not r["few_shot_correct"]:
                print({"ticket": r["text"], "truth": r["true_label"], "zero_shot": r["zero_shot_prediction"], "few_shot": r["few_shot_prediction"], "retrieved": [(s["text"], s["label"]) for s in shots]})
    else:
        print("No underperforming slice found among these predefined slices.")
    (RESULTS / "slice_report.json").write_text(json.dumps(slice_report, indent=2), encoding="utf-8")
else:
    print("Run Section 3 with a valid API key before calculating metrics.")

'''
No underperforming slice was found, as both zero-shot and dynamic few-shot achieved 100% accuracy on the six examples tested. This is likely due to the small sample size,
as only six examples were evaluated using 12 API calls total. Additionally, the similarity between the training and evaluation data may have made the task too easy
for the model to show any noticeable difference in performance. Because of this, a larger evaluation set would be needed to properly determine whether dynamic few-shot
can perform worse than zero-shot, however I only have 20 api calls available so this is not possible for me.
'''

Evaluated: 6
Zero-shot accuracy: 100.0%
Dynamic few-shot accuracy: 100.0%
Change: +0.0 percentage points
{'slice': 'Category: account_access', 'n': 1, 'zero_shot_accuracy': 1.0, 'few_shot_accuracy': 1.0, 'change_pp': 0.0}
{'slice': 'Long ticket (>8 tokens)', 'n': 4, 'zero_shot_accuracy': 1.0, 'few_shot_accuracy': 1.0, 'change_pp': 0.0}
{'slice': 'Retrieved-label majority matches truth', 'n': 6, 'zero_shot_accuracy': 1.0, 'few_shot_accuracy': 1.0, 'change_pp': 0.0}
{'slice': 'Category: billing_issue', 'n': 1, 'zero_shot_accuracy': 1.0, 'few_shot_accuracy': 1.0, 'change_pp': 0.0}
{'slice': 'Category: cancellation_request', 'n': 1, 'zero_shot_accuracy': 1.0, 'few_shot_accuracy': 1.0, 'change_pp': 0.0}
{'slice': 'Short ticket (<=8 tokens)', 'n': 2, 'zero_shot_accuracy': 1.0, 'few_shot_accuracy': 1.0, 'change_pp': 0.0}
{'slice': 'Category: refund_request', 'n': 1, 'zero_shot_accuracy': 1.0, 'few_shot_accuracy': 1.0, 'change_pp': 0.0}
{'slice': 'Category: shipping_delivery', 'n': 1, 'zero_sh

'\nNo underperforming slice was found, as both zero-shot and dynamic few-shot achieved 100% accuracy on the six examples tested. This is likely due to the small sample size,\nas only six examples were evaluated using 12 API calls total. Additionally, the similarity between the training and evaluation data may have made the task too easy \nfor the model to show any noticeable difference in performance. Because of this, a larger evaluation set would be needed to properly determine whether dynamic few-shot \ncan perform worse than zero-shot, however I only have 20 api calls available so this is not possible for me.\n'

#Part 5

In [7]:
if (RESULTS / "evaluation_results.jsonl").exists():
    memo = f"""# Adaptation Decision Memo
\n**Task:** Customer-support ticket classification across six categories.
\n**Dataset and Results:** The dataset contains {len(train)} training examples and {len(eval_)} evaluation examples, although only {len(results)} were evaluated. Zero-shot accuracy was {zero_acc:.1%}, while dynamic few-shot accuracy was {few_acc:.1%}, a change of {(few_acc-zero_acc)*100:+.1f} percentage points.
\n**Prompting:** Improving the instructions given to the AI is a simple and inexpensive way to improve classification. Clearly defining the available categories and requiring a category label as the response can help prevent errors.
\n**Retrieval/RAG:** Retrieving company policies and product documentation can be useful when tickets require additional information. However, this task mainly involves selecting one of six categories, meaning retrieval may provide limited additional benefit.
\n**Dynamic Few-Shot:** Providing similar examples can help the AI understand how tickets should be classified. However, this experiment showed {"an improvement over" if few_acc > zero_acc else "no improvement over" if few_acc == zero_acc else "worse performance than"} zero-shot, so further testing is needed before deciding whether it is worth implementing.
\n**Fine-Tuning:** Fine-tuning may be worth considering if simpler methods continue to produce unsatisfactory results and a larger, higher-quality dataset is available. However, the improvement in accuracy must justify the additional costs of training and maintaining the model.
\n**Doing Nothing:** If zero-shot already provides sufficient accuracy, additional adaptation may not be necessary. Avoiding unnecessary complexity and costs may be the most reasonable decision.
\n**Limitations:** The dataset is synthetic, and only a small portion of the evaluation set was tested. Therefore, these results may not accurately represent performance on real customer-support tickets.
"""
    (RESULTS / "decision_memo.md").write_text(memo, encoding="utf-8")
    display(Markdown(memo))

# Adaptation Decision Memo

**Task:** Customer-support ticket classification across six categories.

**Dataset and Results:** The dataset contains 1200 training examples and 60 evaluation examples, although only 6 were evaluated. Zero-shot accuracy was 100.0%, while dynamic few-shot accuracy was 100.0%, a change of +0.0 percentage points.

**Prompting:** Improving the instructions given to the AI is a simple and inexpensive way to improve classification. Clearly defining the available categories and requiring a category label as the response can help prevent errors.

**Retrieval/RAG:** Retrieving company policies and product documentation can be useful when tickets require additional information. However, this task mainly involves selecting one of six categories, meaning retrieval may provide limited additional benefit.

**Dynamic Few-Shot:** Providing similar examples can help the AI understand how tickets should be classified. However, this experiment showed no improvement over zero-shot, so further testing is needed before deciding whether it is worth implementing.

**Fine-Tuning:** Fine-tuning may be worth considering if simpler methods continue to produce unsatisfactory results and a larger, higher-quality dataset is available. However, the improvement in accuracy must justify the additional costs of training and maintaining the model.

**Doing Nothing:** If zero-shot already provides sufficient accuracy, additional adaptation may not be necessary. Avoiding unnecessary complexity and costs may be the most reasonable decision.

**Limitations:** The dataset is synthetic, and only a small portion of the evaluation set was tested. Therefore, these results may not accurately represent performance on real customer-support tickets.


## Part 4 and 5: decision memo, submit
**TODO (you):** write a short memo weighing prompt, RAG, dynamic few-shot, fine-tuning, and doing nothing for your task, grounded in your numbers. Commit the dataset, update your agent context file, open a pull request with a research note. Rubric: dataset + quality report (30), dynamic few-shot (25), before/after + failure slice (20), decision memo (15), PR hygiene (10).